# 🥔 Potato Disease Classification - Model Comparison

## Overview
This notebook compares multiple pretrained models to find the best one for potato disease detection.

## Models Tested
1. MobileNetV2
2. MobileNetV3-Small
3. MobileNetV3-Large
4. EfficientNetB0
5. DenseNet121
6. NASNetMobile

## Dataset
- **late_blight**: 500 images
- **healthy**: 500 images
- **bacterial_wilt**: 500 images
- **not_potato**: 500 images
- **Total**: 2000 images (balanced dataset)

## Training Strategy
- **Phase 1:** Feature Extraction (frozen base, 10 epochs)
- **Phase 2:** Fine-tuning (unfreeze top layers, 20 epochs)
- **Light augmentation** during training (balanced dataset)

## ⚠️ IMPORTANT
After running this notebook, **REPORT BACK** with the results before proceeding to final training!


In [ ]:
# ========================================
# CELL 1: Mount Drive & Setup
# ========================================

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted!")


In [ ]:
# ========================================
# CELL 2: Imports and Configuration
# ========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import (
    MobileNetV2, MobileNetV3Small, MobileNetV3Large,
    EfficientNetB0, DenseNet121, NASNetMobile
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

# ========================================
# CONFIGURATION
# ========================================

DATASET_PATH = '/content/drive/MyDrive/potato_dataset'
MODEL_PATH = '/content/drive/MyDrive/models'
os.makedirs(MODEL_PATH, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 20
PHASE1_LR = 0.001
PHASE2_LR = 0.0001
UNFREEZE_LAYERS = 30

CLASS_NAMES = ['late_blight', 'healthy', 'bacterial_wilt', 'not_potato']
NUM_CLASSES = 4

print("🥔 POTATO DISEASE CLASSIFIER - MODEL COMPARISON")
print("=" * 60)
print(f"Dataset: {DATASET_PATH}")
print(f"Model Path: {MODEL_PATH}")
print(f"Image Size: {IMG_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Phase 1: {PHASE1_EPOCHS} epochs (frozen base)")
print(f"Phase 2: {PHASE2_EPOCHS} epochs (fine-tuning)")
print(f"Classes: {CLASS_NAMES}")

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU available: {gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("\n⚠️ No GPU available - using CPU")


In [ ]:
# ========================================
# CELL 3: Analyze Dataset Distribution & Clean Corrupted Images
# ========================================

from PIL import Image, ImageFile
import warnings
warnings.filterwarnings('ignore')

# Allow loading of truncated images (but we'll still remove corrupted ones)
ImageFile.LOAD_TRUNCATED_IMAGES = True

print("📊 Analyzing dataset distribution...")

# First, clean corrupted images
print("\n🧹 Cleaning corrupted images...")
corrupted_count = 0

for class_name in CLASS_NAMES:
    class_path = os.path.join(DATASET_PATH, class_name)
    if not os.path.exists(class_path):
        continue
    
    images = [f for f in os.listdir(class_path) 
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    for img_file in images:
        img_path = os.path.join(class_path, img_file)
        try:
            # Try to open and verify image
            with Image.open(img_path) as img:
                img.verify()  # Verify image integrity
            
            # Reopen for actual loading (verify closes the file)
            img = Image.open(img_path)
            img.load()  # Try to load the image data
            img.close()
            
        except Exception as e:
            print(f"   🗑️ Removing corrupted: {class_name}/{img_file}")
            try:
                os.remove(img_path)
                corrupted_count += 1
            except:
                pass

print(f"✅ Cleaned {corrupted_count} corrupted images\n")

# Now count valid images
class_counts = {}
for class_name in CLASS_NAMES:
    class_path = os.path.join(DATASET_PATH, class_name)
    if os.path.exists(class_path):
        images = [f for f in os.listdir(class_path) 
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        class_counts[class_name] = len(images)
    else:
        class_counts[class_name] = 0
        print(f"⚠️ Warning: {class_path} not found")

print("\n📊 Class Distribution (after cleaning):")
for class_name, count in class_counts.items():
    print(f"   {class_name}: {count} images")

total = sum(class_counts.values())
print(f"\n   Total: {total} images")

# Plot distribution
plt.figure(figsize=(10, 6))
plt.bar(class_counts.keys(), class_counts.values(), color=['#FF6B6B', '#4ECDC4', '#FFE66D', '#95E1D3'])
plt.title('Potato Dataset Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Class', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

if total == 0:
    raise ValueError("❌ No images found! Please check your dataset path.")


In [ ]:
# ========================================
# CELL 4: Create Train/Val/Test Split (80/10/10)
# ========================================

import shutil
from sklearn.model_selection import train_test_split

print("📁 Creating train/val/test split (80/10/10)...")

# Organize path
ORGANIZED_PATH = os.path.join(DATASET_PATH, 'organized')

# Remove existing organized folder if it exists (to avoid mixing old data)
if os.path.exists(ORGANIZED_PATH):
    import shutil
    shutil.rmtree(ORGANIZED_PATH)
    print("   🗑️ Removed existing organized folder")

for split in ['train', 'val', 'test']:
    for class_name in CLASS_NAMES:
        os.makedirs(os.path.join(ORGANIZED_PATH, split, class_name), exist_ok=True)

# Split data (with additional validation to skip corrupted images)
for class_name in CLASS_NAMES:
    class_path = os.path.join(DATASET_PATH, class_name)
    if not os.path.exists(class_path):
        continue
    
    # Get all images and validate them
    valid_images = []
    for img_file in os.listdir(class_path):
        if not img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            continue
        
        img_path = os.path.join(class_path, img_file)
        try:
            # Quick validation
            with Image.open(img_path) as img:
                img.verify()
            valid_images.append(img_file)
        except:
            print(f"   ⚠️ Skipping corrupted image during split: {class_name}/{img_file}")
            continue
    
    if len(valid_images) == 0:
        print(f"   ⚠️ No valid images found for {class_name}")
        continue
    
    # Split: 80% train, 10% val, 10% test
    train_files, temp_files = train_test_split(valid_images, test_size=0.2, random_state=42)
    val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)
    
    # Copy files
    for split_name, files in [('train', train_files), ('val', val_files), ('test', test_files)]:
        for img_file in files:
            src = os.path.join(class_path, img_file)
            dst = os.path.join(ORGANIZED_PATH, split_name, class_name, img_file)
            try:
                shutil.copy2(src, dst)
            except Exception as e:
                print(f"   ⚠️ Error copying {img_file}: {str(e)}")
                continue
    
    print(f"   {class_name}: {len(train_files)} train, {len(val_files)} val, {len(test_files)} test")

print(f"\n✅ Dataset organized at: {ORGANIZED_PATH}")


In [ ]:
# ========================================
# CELL 5: Create Data Generators
# ========================================

print("📊 Creating Data Generators...") 

# Light augmentation for balanced dataset
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.9, 1.1],
    zoom_range=0.1
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

# Create generators
train_generator = train_datagen.flow_from_directory(
    os.path.join(ORGANIZED_PATH, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=True,
    seed=42
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(ORGANIZED_PATH, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(ORGANIZED_PATH, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

print(f"\n✅ Generators Created:")
print(f"   Train: {train_generator.samples} samples, {len(train_generator)} batches")
print(f"   Val: {val_generator.samples} samples, {len(val_generator)} batches")
print(f"   Test: {test_generator.samples} samples, {len(test_generator)} batches")
print(f"\n   Class indices: {train_generator.class_indices}")

# Calculate class weights (should be balanced, so weights close to 1.0)
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))

print("\n⚖️ Class Weights:")
for name, idx in train_generator.class_indices.items():
    print(f"   {name}: {class_weight_dict[idx]:.3f}")


In [ ]:
# ========================================
# CELL 6: Define Models to Test
# ========================================

def build_model(base_model_class, input_shape, num_classes, model_name):
    """Build model with custom head"""
    base_model = base_model_class(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_model.trainable = False
    
    # Build model
    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    
    model = Model(inputs, outputs, name=model_name)
    return model, base_model

# Models to test
MODELS_TO_TEST = {
    'MobileNetV2': {
        'base': MobileNetV2,
        'unfreeze_layers': 30
    },
    'MobileNetV3Small': {
        'base': MobileNetV3Small,
        'unfreeze_layers': 30
    },
    'MobileNetV3Large': {
        'base': MobileNetV3Large,
        'unfreeze_layers': 30
    },
    'EfficientNetB0': {
        'base': EfficientNetB0,
        'unfreeze_layers': 50
    },
    'DenseNet121': {
        'base': DenseNet121,
        'unfreeze_layers': 50
    },
    'NASNetMobile': {
        'base': NASNetMobile,
        'unfreeze_layers': 30
    }
}

print("🏗️ Models to test:")
for model_name in MODELS_TO_TEST.keys():
    print(f"   - {model_name}")


In [ ]:
# ========================================
# CELL 7: Training Function
# ========================================

def train_and_evaluate_model(model_name, model_config, train_gen, val_gen, test_gen, class_weights):
    """Train and evaluate a single model"""
    
    print(f"\n{'='*60}")
    print(f"🎯 Training: {model_name}")
    print(f"{'='*60}")
    
    start_time = time.time()
    
    # Build model
    model, base_model = build_model(
        model_config['base'],
        IMG_SIZE + (3,),
        NUM_CLASSES,
        model_name
    )
    
    # Compile
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=PHASE1_LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    print(f"   Total params: {model.count_params():,}")
    
    # ========== PHASE 1: Feature Extraction ==========
    print(f"\n📌 Phase 1: Feature Extraction ({PHASE1_EPOCHS} epochs)...")
    
    callbacks_phase1 = [
        EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
    ]
    
    history1 = model.fit(
        train_gen,
        epochs=PHASE1_EPOCHS,
        validation_data=val_gen,
        class_weight=class_weights,
        callbacks=callbacks_phase1,
        verbose=1
    )
    
    # ========== PHASE 2: Fine-tuning ==========
    print(f"\n📌 Phase 2: Fine-tuning ({PHASE2_EPOCHS} epochs)...")
    
    # Unfreeze top layers
    base_model.trainable = True
    unfreeze_count = model_config['unfreeze_layers']
    total_layers = len(base_model.layers)
    
    for layer in base_model.layers[:-unfreeze_count]:
        layer.trainable = False
    
    # Recompile
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=PHASE2_LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    trainable = sum([tf.reduce_prod(v.shape).numpy() for v in model.trainable_variables])
    print(f"   Trainable params: {trainable:,}")
    
    callbacks_phase2 = [
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-8, verbose=1)
    ]
    
    history2 = model.fit(
        train_gen,
        epochs=PHASE2_EPOCHS,
        validation_data=val_gen,
        class_weight=class_weights,
        callbacks=callbacks_phase2,
        verbose=1
    )
    
    # ========== EVALUATION ==========
    print(f"\n📊 Evaluating {model_name}...")
    
    try:
        test_gen.reset()
        test_loss, test_acc = model.evaluate(test_gen, verbose=0)
        
        # Get predictions for detailed analysis
        test_gen.reset()
        predictions = model.predict(test_gen, verbose=0)
        predicted_classes = np.argmax(predictions, axis=1)
        true_classes = test_gen.classes
        
        # Average confidence
        avg_confidence = np.mean(np.max(predictions, axis=1))
        
        # Per-class accuracy
        cm = confusion_matrix(true_classes, predicted_classes)
        per_class_acc = cm.diagonal() / cm.sum(axis=1)
        
        # Model size (approximate)
        model_size_mb = model.count_params() * 4 / (1024 * 1024)  # Rough estimate
        
        training_time = (time.time() - start_time) / 60  # minutes
        
        results = {
            'model_name': model_name,
            'test_accuracy': test_acc,
            'avg_confidence': avg_confidence,
            'training_time': training_time,
            'model_size_mb': model_size_mb,
            'per_class_accuracy': {
                CLASS_NAMES[i]: float(per_class_acc[i]) for i in range(len(CLASS_NAMES))
            }
        }
        
        print(f"   ✅ Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
        print(f"   ✅ Avg Confidence: {avg_confidence:.4f}")
        print(f"   ⏱️ Training Time: {training_time:.1f} minutes")
        
        return results
        
    except Exception as e:
        print(f"   ❌ Evaluation error: {str(e)}")
        # Return partial results
        training_time = (time.time() - start_time) / 60
        return {
            'model_name': model_name,
            'test_accuracy': 0.0,
            'avg_confidence': 0.0,
            'training_time': training_time,
            'model_size_mb': model.count_params() * 4 / (1024 * 1024),
            'per_class_accuracy': {cls: 0.0 for cls in CLASS_NAMES},
            'error': str(e)
        }


In [ ]:
# ========================================
# CELL 8: Run Model Comparison
# ========================================

print("\n" + "="*70)
print("🚀 STARTING MODEL COMPARISON")
print("="*70)

all_results = []

for model_name, model_config in MODELS_TO_TEST.items():
    try:
        results = train_and_evaluate_model(
            model_name, model_config,
            train_generator, val_generator, test_generator,
            class_weight_dict
        )
        all_results.append(results)
    except Exception as e:
        print(f"\n❌ Error training {model_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue

print("\n" + "="*70)
print("✅ MODEL COMPARISON COMPLETE!")
print("="*70)


In [ ]:
# ========================================
# CELL 9: Results Summary Table
# ========================================

print("\n" + "="*70)
print("📊 MODEL COMPARISON RESULTS - SHARE THIS WITH ME!")
print("="*70)

# Create DataFrame
df_data = []
for r in all_results:
    row = {
        'Model': r['model_name'],
        'Test Accuracy': f"{r['test_accuracy']:.4f}",
        'Avg Confidence': f"{r['avg_confidence']:.4f}",
        'Time (min)': f"{r['training_time']:.1f}",
        'Size (MB)': f"{r['model_size_mb']:.1f}"
    }
    # Add per-class accuracy
    for class_name in CLASS_NAMES:
        row[class_name] = f"{r['per_class_accuracy'][class_name]:.4f}"
    df_data.append(row)

df = pd.DataFrame(df_data)

# Sort by accuracy
df['Test Accuracy Float'] = df['Test Accuracy'].astype(float)
df = df.sort_values('Test Accuracy Float', ascending=False)
df = df.drop('Test Accuracy Float', axis=1)

print("\n📋 SUMMARY TABLE:")
print(df.to_string(index=False))

# Find best model
best_result = max(all_results, key=lambda x: x['test_accuracy'])
print(f"\n🏆 BEST MODEL: {best_result['model_name']}")
print(f"   Test Accuracy: {best_result['test_accuracy']:.4f} ({best_result['test_accuracy']*100:.2f}%)")
print(f"   Avg Confidence: {best_result['avg_confidence']:.4f}")
print(f"   Model Size: {best_result['model_size_mb']:.1f} MB")

# Save results
results_path = os.path.join(MODEL_PATH, 'potato_comparison_results.csv')
df.to_csv(results_path, index=False)
print(f"\n💾 Results saved to: {results_path}")


In [ ]:
# ========================================
# CELL 10: Visualize Comparison Results
# ========================================

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Accuracy comparison
ax1 = axes[0, 0]
models = [r['model_name'] for r in all_results]
accuracies = [r['test_accuracy'] for r in all_results]
colors = ['#FF6B6B' if acc == max(accuracies) else '#95E1D3' for acc in accuracies]
ax1.barh(models, accuracies, color=colors)
ax1.set_xlabel('Test Accuracy', fontsize=12)
ax1.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax1.set_xlim([0.8, 1.0])
ax1.grid(axis='x', alpha=0.3)
for i, (model, acc) in enumerate(zip(models, accuracies)):
    ax1.text(acc + 0.005, i, f'{acc:.3f}', va='center', fontweight='bold')

# 2. Confidence comparison
ax2 = axes[0, 1]
confidences = [r['avg_confidence'] for r in all_results]
colors = ['#FF6B6B' if conf == max(confidences) else '#95E1D3' for conf in confidences]
ax2.barh(models, confidences, color=colors)
ax2.set_xlabel('Average Confidence', fontsize=12)
ax2.set_title('Model Confidence Comparison', fontsize=14, fontweight='bold')
ax2.set_xlim([0.8, 1.0])
ax2.grid(axis='x', alpha=0.3)
for i, (model, conf) in enumerate(zip(models, confidences)):
    ax2.text(conf + 0.005, i, f'{conf:.3f}', va='center', fontweight='bold')

# 3. Per-class accuracy heatmap
ax3 = axes[1, 0]
per_class_data = []
for r in all_results:
    row = [r['per_class_accuracy'][cls] for cls in CLASS_NAMES]
    per_class_data.append(row)
per_class_df = pd.DataFrame(per_class_data, index=models, columns=CLASS_NAMES)
sns.heatmap(per_class_df, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax3, cbar_kws={'label': 'Accuracy'})
ax3.set_title('Per-Class Accuracy Heatmap', fontsize=14, fontweight='bold')
ax3.set_ylabel('Model', fontsize=12)

# 4. Model size vs accuracy
ax4 = axes[1, 1]
sizes = [r['model_size_mb'] for r in all_results]
scatter = ax4.scatter(sizes, accuracies, s=200, c=confidences, cmap='viridis', alpha=0.7, edgecolors='black')
for i, model in enumerate(models):
    ax4.annotate(model, (sizes[i], accuracies[i]), xytext=(5, 5), textcoords='offset points', fontsize=9)
ax4.set_xlabel('Model Size (MB)', fontsize=12)
ax4.set_ylabel('Test Accuracy', fontsize=12)
ax4.set_title('Model Size vs Accuracy', fontsize=14, fontweight='bold')
ax4.grid(alpha=0.3)
plt.colorbar(scatter, ax=ax4, label='Confidence')

plt.tight_layout()
chart_path = os.path.join(MODEL_PATH, 'potato_comparison_charts.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"📊 Charts saved to: {chart_path}")


# ⛔ STOP HERE - REPORT BACK!

## 📋 Please share the results summary with me:

Copy and paste the output from **CELL 9** which shows:
- Summary table with all model results
- Best model name and its performance metrics

After you report the results, I'll create the final training notebook using the best performing model!

---

## Expected Output Format:
```
📊 MODEL COMPARISON RESULTS - SHARE THIS WITH ME!
======================================================================

📋 SUMMARY TABLE:
Model             Test Accuracy  Avg Confidence  Time (min)  Size (MB)  ...
MobileNetV2       0.9850         0.9820          45.2        20.1       ...
DenseNet121       0.9930         0.9880          120.5       32.0       ...
...

🏆 BEST MODEL: DenseNet121
   Test Accuracy: 0.9930 (99.30%)
   Avg Confidence: 0.9880
   Model Size: 32.0 MB
```
